# 02 - Hiperparámetros de Isolation Forest, Efecto Swamping/Masking y Calibración de Umbrales

## Curso de Machine Learning: Modelos No Supervisados

En este cuaderno profundizaremos en la mecánica interna y el ajuste fino de los hiperparámetros de `IsolationForest` en Scikit-Learn. Comprender cómo interactúan el tamaño de submuestra (`max_samples`), el número de árboles (`n_estimators`) y la proporción esperada de fraude (`contamination`) es determinante para evitar falsas alarmas masivas en banca y comercio electrónico.

---

### Contenidos
1. Los Fenómenos de Enmascaramiento (*Masking*) y Empantanamiento (*Swamping*)
2. Por qué Menos es Más en Isolation Forest: El Hiperparámetro `max_samples` ($\psi = 256$)
3. El Rol Exacto de `contamination`: ¿Cambia los Árboles o Solo Desplaza el Umbral?
4. Convergencia de la Varianza con `n_estimators`
5. Subespacios Aleatorios de Características: `max_features` para Alta Dimensionalidad
6. Calibración de Umbrales Operativos por Presupuesto de Auditoría (Top-$K$ Alertas)
7. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Masking y Swamping: Por qué `max_samples=256` Supera a Usar Todos los Datos

En casi todo el Machine Learning, entrenar con más datos por árbol mejora el modelo. **En Isolation Forest ocurre exactamente lo contrario**: entrenar cada árbol con decenas de miles de muestras degrada la detección debido a dos patologías:
- **Masking (Enmascaramiento)**: Cuando hay demasiadas anomalías agrupadas juntas en la muestra del árbol, se protegen entre sí y requieren muchos cortes para separarse, pareciendo puntos normales.
- **Swamping (Empantanamiento)**: Cuando hay demasiados puntos normales rodeando a una anomalía, los cortes aleatorios se saturan dividiendo la nube normal antes de aislar la anomalía.

Al restringir `max_samples = 256` (sin reemplazo), la probabilidad de que un cluster de fraude pequeño contamine la muestra del árbol es mínima, permitiendo aislar anomalías agrupadas con suma facilidad.


In [ ]:
# Experimento: Un cluster pequeño de fraude organizado (Masking) evaluado con distintos max_samples
rng = np.random.default_rng(42)
X_legit = rng.normal(0, 1.2, size=(4000, 4))
# Ataque coordinado de bots (60 transacciones fraudulentas muy parecidas entre si)
X_bot_attack = rng.normal(4.2, 0.25, size=(60, 4))

X_exp = np.vstack([X_legit, X_bot_attack])
y_fraude_bin = np.array([0]*4000 + [1]*60)

tamanos_psi = [64, 128, 256, 512, 1024, 2048, 4060]
aucs_psi = []

for psi in tamanos_psi:
    modelo = IsolationForest(n_estimators=100, max_samples=psi, random_state=42).fit(X_exp)
    # Invertimos el signo para que mayor score = mayor probabilidad de fraude
    anomaly_score = -modelo.score_samples(X_exp)
    aucs_psi.append(roc_auc_score(y_fraude_bin, anomaly_score))

plt.figure(figsize=(10, 4.8))
plt.plot(tamanos_psi, aucs_psi, 'o-', color='#d62728', lw=2.5)
plt.axvline(256, color='#1f77b4', linestyle='--', lw=2, label='Estándar Óptimo (max_samples = 256)')
plt.xscale('log', base=2)
plt.xticks(tamanos_psi, labels=[str(p) for p in tamanos_psi])
plt.xlabel('Tamaño de Submuestra por Árbol (max_samples - Escala Log2)', fontsize=11, fontweight='bold')
plt.ylabel('ROC-AUC en Detección del Ataque Coordinado', fontsize=11, fontweight='bold')
plt.title('Efecto Masking: Usar Todas las Muestras (4060) Hunde la Detección de Fraude Agrupado', fontsize=12, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## 2. El Hiperparámetro `contamination`: ¿Qué Hace Realmente?

Un mito muy extendido es creer que el parámetro `contamination` altera cómo se construyen los árboles de aislamiento. **Eso es falso.**

Los árboles iTrees se construyen de manera $100\%$ ciega e idéntica sin importar el valor de `contamination`. Lo único que hace `contamination = c` es fijar el percentil $100 \cdot c$ sobre los puntajes de entrenamiento para establecer el atributo `offset_` que separa las etiquetas `-1` de `+1` en `.predict(X)`:

$$\text{decision\_function}(x) = \text{score\_samples}(x) - \text{offset\_}$$


In [ ]:
# Demostración de que score_samples(X) es idéntico para cualquier valor de contamination
if_c01 = IsolationForest(contamination=0.01, random_state=42).fit(X_exp)
if_c20 = IsolationForest(contamination=0.20, random_state=42).fit(X_exp)

diff_scores = np.max(np.abs(if_c01.score_samples(X_exp) - if_c20.score_samples(X_exp)))
print(f"Diferencia en score_samples entre contamination=0.01 y contamination=0.20: {diff_scores:.1f}")
print(f"Offset interno con c=0.01: {if_c01.offset_:.4f} -> Alertas disparadas: {(if_c01.predict(X_exp)==-1).sum()}")
print(f"Offset interno con c=0.20: {if_c20.offset_:.4f} -> Alertas disparadas: {(if_c20.predict(X_exp)==-1).sum()}")


## 3. Calibración por Presupuesto Operativo (Capacidad de Analistas de Fraude)

En un banco real no sabemos de antemano si el fraude de hoy es del $0.4\%$ o del $1.2\%$. Sin embargo, sí sabemos **cuántas alertas diarias puede revisar nuestro equipo humano de analistas antifraude** (ej. presupuesto operativo $B = 50$ casos más riesgosos del día).

La práctica recomendada en producción consiste en usar directamente `-model.score_samples(X)` para ordenar las transacciones de mayor a menor riesgo y cortar por capacidad operativa o umbral de costo-beneficio.


In [ ]:
# Curva de Captura de Fraude (Recall) y Precisión según el Presupuesto de Revisión Diaria
scores_riesgo = -if_c01.score_samples(X_exp)
presupuestos = range(10, 200, 10)
precisiones = []
recalls = []

orden_riesgo = np.argsort(scores_riesgo)[::-1]
for b in presupuestos:
    pred_top_b = np.zeros_like(y_fraude_bin)
    pred_top_b[orden_riesgo[:b]] = 1
    precisiones.append(precision_score(y_fraude_bin, pred_top_b) * 100)
    recalls.append(recall_score(y_fraude_bin, pred_top_b) * 100)

plt.figure(figsize=(10, 5))
plt.plot(presupuestos, recalls, 's-', color='#2ca02c', lw=2.2, label='% del Fraude Total Capturado (Recall)')
plt.plot(presupuestos, precisiones, 'o-', color='#1f77b4', lw=2.2, label='% de Acierto en las Alertas Revisadas (Precision)')
plt.axvline(60, color='gray', linestyle='--', label='Número Real de Fraudes (60)')
plt.xlabel('Presupuesto Diario de Alertas Revisadas (Top-K Transacciones)', fontsize=11, fontweight='bold')
plt.ylabel('Porcentaje (%)', fontsize=11, fontweight='bold')
plt.title('Curva Operativa de Auditoría Antifraude Guiada por Isolation Forest', fontsize=12, fontweight='bold')
plt.legend(fontsize=10.5)
plt.tight_layout()
plt.show()


## 4. Conclusiones

1. **`max_samples=256`**: Es el valor óptimo que previene simultáneamente el *masking* y el *swamping*.
2. **`contamination`**: Solo desplaza el umbral de corte (`offset_`). Si deseas usar el umbral heurístico original del artículo de Liu et al. ($s = 0.5$), configura `contamination='auto'`.
3. **Ranking Continuo**: En sistemas antifraude reales, utiliza siempre `-model.score_samples(X)` para priorizar la cola de investigación.
